In [2]:
!pip -q install ragas langchain pandas tqdm
!pip -q install langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.3/284.3 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [6]:
from langchain_community.chat_models import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

avalai_api_key = "aa-tkxSAYzHstC7EJDuOLnW69hzYQh6KlDp8bG1bM9ZzB7jIFm0"
avalai_base_url = "https://api.avalai.ir/v1"
avalai_model = "gpt-5-nano"

# ساخت LLM با provider سفارشی
lc_llm = ChatOpenAI(
    openai_api_key=avalai_api_key,
    openai_api_base=avalai_base_url,
    model=avalai_model,
    temperature=0.0
)

ragas_llm = LangchainLLMWrapper(lc_llm)


In [7]:
import json, os
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

images_dir = "/content/drive/MyDrive/Research/Wageningen University /Datasets/soil health dataset"  # مسیر فولدر عکس‌ها
json_path = "/content/drive/MyDrive/Research/Wageningen University /Datasets/Output SoilHealth/rdf_extractions_fewShot.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

samples = []
for item in data.get("dataset", []):
    img_name = item.get("source_image")
    rdf_text = item.get("rdf_graph_turtle", "")
    img_path = os.path.join(images_dir, img_name)

    if not os.path.exists(img_path):
        print("⚠️ تصویر پیدا نشد:", img_path)
        continue

    sample = SingleTurnSample(
        user_input=f"Extract RDF triples from this image ({img_name})",
        response=rdf_text,
        retrieved_contexts=[img_path],  # خیلی مهم برای متریک‌های مولتی‌مودال
        metadata={"image": img_name}
    )
    samples.append(sample)

dataset = EvaluationDataset(samples=samples)
print("تعداد نمونه‌ها:", len(samples))


⚠️ تصویر پیدا نشد: /content/drive/MyDrive/Research/Wageningen University /Datasets/soil health dataset/table_1.6.jpg
⚠️ تصویر پیدا نشد: /content/drive/MyDrive/Research/Wageningen University /Datasets/soil health dataset/table_1.1.jpg
⚠️ تصویر پیدا نشد: /content/drive/MyDrive/Research/Wageningen University /Datasets/soil health dataset/table_8.2.jpg
⚠️ تصویر پیدا نشد: /content/drive/MyDrive/Research/Wageningen University /Datasets/soil health dataset/table_2.9.jpg
⚠️ تصویر پیدا نشد: /content/drive/MyDrive/Research/Wageningen University /Datasets/soil health dataset/table_5.4.jpg
⚠️ تصویر پیدا نشد: /content/drive/MyDrive/Research/Wageningen University /Datasets/soil health dataset/table_1.2-checkpoint.jpg
تعداد نمونه‌ها: 57


In [8]:
from ragas import evaluate
from ragas.metrics import MultiModalFaithfulness, MultiModalRelevance

metrics = [MultiModalFaithfulness(), MultiModalRelevance()]

results = evaluate(
    dataset,
    metrics=metrics,
    llm=ragas_llm,
    show_progress=True
)

df = results.to_pandas()
df.head()


Evaluating:   0%|          | 0/114 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[35]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for requests.', 'type': 'rate_limit_exceeded', 'param': None, 'code': 'rate_limit_exceeded', 'solution': 'Pace your requests or upgrade to higher levels. Read the Rate limit guide at https://chat.avalai.ir/platform/limits or contact support at support@avalai.ir and include the request ID 01997acf-566f-7e80-9f1a-d08bbb60dfa0 in your email if you believe this is an error.', 'request_id': '01997acf-566f-7e80-9f1a-d08bbb60dfa0'}})


,user_input,retrieved_contexts,response,faithful_rate,relevance_rate
0,Extract RDF triples from this image (table_10....,[/content/drive/MyDrive/Research/Wageningen Un...,@prefix she: <https://soilwise-he.github.io/s...,0.0,1.0
1,Extract RDF triples from this image (table_7.3...,[/content/drive/MyDrive/Research/Wageningen Un...,@prefix she: <https://soilwise-he.github.io/s...,1.0,1.0
2,Extract RDF triples from this image (table_6.2...,[/content/drive/MyDrive/Research/Wageningen Un...,@prefix she: <https://soilwise-he.github.io/s...,1.0,1.0
3,Extract RDF triples from this image (table_4.2...,[/content/drive/MyDrive/Research/Wageningen Un...,@prefix she: <https://soilwise-he.github.io/s...,1.0,1.0
4,Extract RDF triples from this image (table_9.4...,[/content/drive/MyDrive/Research/Wageningen Un...,@prefix she: <https://soilwise-he.github.io/s...,1.0,1.0


from matplotlib import pyplot as plt
_df_0['faithful_rate'].plot(kind='hist', bins=20, title='faithful_rate')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_1.groupby('user_input').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
plt.subplots(figsize=(8, 8))
df_2dhist = pd.DataFrame({
    x_label: grp['response'].value_counts()
    for x_label, grp in _df_8.groupby('user_input')
})
sns.heatmap(df_2dhist, cmap='viridis')
plt.xlabel('user_input')
_ = plt.ylabel('response')

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_9['user_input'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_9, x='faithful_rate', y='user_input', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_10['response'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_10, x='faithful_rate', y='response', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [13]:
import pandas as pd

# فایل per_sample_scores را بخوان
df = pd.read_csv("/content/per_sample_scores.csv")

# محاسبه میانگین‌ها
summary = df[["faithful_rate", "relevance_rate"]].mean().reset_index()
summary.columns = ["metric", "mean_score"]

# ذخیره
summary.to_csv("/content/summary.csv", index=False)

summary


,metric,mean_score
0,faithful_rate,0.754386
1,relevance_rate,0.964912
